In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy

In [2]:
import scienceplots

plt.style.use('science')
plt.rcParams.update({
    'text.usetex': True,
    'text.latex.preamble': r'\usepackage[T1]{fontenc} \usepackage{polski} \usepackage[utf8]{inputenc}'
})
figsize = (6, 3)
new_orange = '#ff5500'

#### Zadanie 1.
W ankiecie przedstawionej na poprzedniej liście pracownicy zostali poproszeni o wyrażenie opinii na temat skuteczności szkolenia "Efektywna komunikacja w zespole" zorganizowanego przez firmę. Wśród próbki 200 pracowników (losowanie proste ze zwracaniem) uzyskano wyniki:

- 14 pracowników - bardzo niezadowolonych,
- 17 pracowników - niezadowolonych,
- 40 pracowników - nie ma zdania,
- 100 pracowników - zadowolonych,
- 29 pracowników - bardzo zadowolonych.

Na podstawie danych wyznacz przedział ufności dla wektora prawdopodobieństw opisującego stopień zadowolenia ze szkolenia. Przyjmij poziom ufności 0.95.

In [3]:
n = 200
counts = np.array([14, 17, 40, 100, 29])
k = len(counts)
labels = [
    'Bardzo niezadowoleni',
    'Niezadowoleni',
    'Nie ma zdania',
    'Zadowoleni',
    'Bardzo zadowoleni'
]

p_hat = counts / n

alpha = 0.05
z_val = scipy.stats.norm.ppf(1 - alpha / (2 * k))

se = np.sqrt((p_hat * (1 - p_hat)) / n)
margin_of_error = z_val * se

ci_lower = np.maximum(0, p_hat - margin_of_error)
ci_upper = np.minimum(1, p_hat + margin_of_error)

df_results = pd.DataFrame({
    'Opinia': labels,
    'n_i': counts,
    'p_hat': p_hat,
    'Dolna granica (95%)': ci_lower.round(4),
    'Górna granica (95%)': ci_upper.round(4)
})

print("Tabela przedziałów ufności:")
print(df_results.to_string(index=False))

Tabela przedziałów ufności:
              Opinia  n_i  p_hat  Dolna granica (95%)  Górna granica (95%)
Bardzo niezadowoleni   14  0.070               0.0235               0.1165
       Niezadowoleni   17  0.085               0.0342               0.1358
       Nie ma zdania   40  0.200               0.1271               0.2729
          Zadowoleni  100  0.500               0.4089               0.5911
   Bardzo zadowoleni   29  0.145               0.0809               0.2091


### Zadanie 3

In [4]:
def p_value_multinomial(x, p0):
    x = np.array(x)
    p0 = np.array(p0)

    if len(x) != len(p0):
        raise ValueError("Błąd: Wektor obserwacji x i wektor prawdopodobieństw p0 muszą mieć taką samą długość.")

    n = np.sum(x)
    k = len(x)
    df = k - 1

    E = n * p0

    chi2_pearson = np.sum((x - E) ** 2 / E)
    p_val_pearson = scipy.stats.chi2.sf(chi2_pearson, df)

    idx = x > 0
    chi2_lr = 2 * np.sum(x[idx] * np.log(x[idx] / E[idx]))
    p_val_lr = scipy.stats.chi2.sf(chi2_lr, df)

    wyniki = pd.DataFrame({
        'Statystyka': [chi2_pearson, chi2_lr],
        'p-value': [p_val_pearson, p_val_lr],
        'Stopnie swobody': [df, df]
    }, index=['Chi-kwadrat Pearsona', 'Iloraz wiarogodności (G-test)'])

    return wyniki


df = pd.read_csv('dane.csv', sep=';')

print("Dostępne działy w pliku:", df['DZIAŁ'].unique())

nazwa_dzialu = 'PD'
df_dzial = df[df['DZIAŁ'] == nazwa_dzialu]

mozliwe_odpowiedzi = [-2, -1, 0, 1, 2]

# value_counts() zlicza odpowiedzi, a reindex zapewnia, że braki zostaną uzupełnione 0.
zliczenia = df_dzial['PYT_1'].value_counts().reindex(mozliwe_odpowiedzi, fill_value=0).values

prawdopodobienstwa_H0 = [0.2, 0.2, 0.2, 0.2, 0.2]
alpha = 0.05

print(f"Analiza dla działu: {nazwa_dzialu}")
print(f"Liczebności odpowiedzi dla kategorii [-2, -1, 0, 1, 2]: {zliczenia}")
print("-" * 50)

wyniki_testu = p_value_multinomial(x=zliczenia, p0=prawdopodobienstwa_H0)
print(wyniki_testu)
print("-" * 50)

p_val = wyniki_testu.loc['Chi-kwadrat Pearsona', 'p-value']

print(f"Poziom istotności (alpha) = {alpha}")
print(f"Otrzymane p-value (Pearson) = {p_val:.4f}")

if p_val < alpha:
    print("Wniosek: Odrzucamy hipotezę zerową (H0).")
    print("Rozkład odpowiedzi NIE jest równomierny.")
else:
    print("Wniosek: Nie ma podstaw do odrzucenia hipotezy zerowej (H0).")
    print("Możemy przyjąć, że rozkład odpowiedzi jest równomierny.")

Dostępne działy w pliku: ['IT' 'PD' 'MK' 'HR']
Analiza dla działu: PD
Liczebności odpowiedzi dla kategorii [-2, -1, 0, 1, 2]: [ 9 10 17 51 11]
--------------------------------------------------
                               Statystyka       p-value  Stopnie swobody
Chi-kwadrat Pearsona            64.857143  2.757834e-13                4
Iloraz wiarogodności (G-test)   52.527114  1.070199e-10                4
--------------------------------------------------
Poziom istotności (alpha) = 0.05
Otrzymane p-value (Pearson) = 0.0000
Wniosek: Odrzucamy hipotezę zerową (H0).
Rozkład odpowiedzi NIE jest równomierny.


### zadanie 7

In [5]:
def lr_test_independence(observed_table):
    """
    Oblicza statystykę G oraz p-value dla testu niezależności opartego
    na ilorazie wiarogodności dla tablicy dwudzielczej.
    """
    O = np.array(observed_table)

    row_sums = O.sum(axis=1)
    col_sums = O.sum(axis=0)
    total = O.sum()

    E = np.outer(row_sums, col_sums) / total

    df = (O.shape[0] - 1) * (O.shape[1] - 1)

    mask = O > 0
    G = 2 * np.sum(O[mask] * np.log(O[mask] / E[mask]))

    p_value = scipy.stats.chi2.sf(G, df)

    return G, p_value, df


df = pd.read_csv('ankieta2.csv', sep=';')
df = df.rename(columns={'P£E∆': 'PŁEĆ', 'STAĮ': 'STAŻ', 'DZIA£': 'DZIAŁ'})

tablica_krzyzowa = pd.crosstab(df['PYT_2'], df['CZY_KIER'])

print("--- Tablica krzyżowa (Obserwacje) ---")
print(tablica_krzyzowa)
print("-" * 50)

G_stat, p_val_lr, stopnie_swobody = lr_test_independence(tablica_krzyzowa)

print("--- Wyniki Testu Ilorazu Wiarogodności (G-test) ---")
print(f"Statystyka G: {G_stat:.4f}")
print(f"Stopnie swobody: {stopnie_swobody}")
print(f"p-value:      {p_val_lr:.4f}")
print("-" * 50)

alpha = 0.05
if p_val_lr < alpha:
    print("Wniosek: Odrzucamy H0.")
    print("Zmienne PYT_2 i CZY_KIER są ZALEŻNE.")
else:
    print("Wniosek: Brak podstaw do odrzucenia H0.")
    print("Zmienne PYT_2 i CZY_KIER są NIEZALEŻNE.")


--- Tablica krzyżowa (Obserwacje) ---
CZY_KIER  Nie  Tak
PYT_2             
-2         64   10
-1         18    2
 1          0    2
 2         91   13
--------------------------------------------------
--- Wyniki Testu Ilorazu Wiarogodności (G-test) ---
Statystyka G: 8.3285
Stopnie swobody: 3
p-value:      0.0397
--------------------------------------------------
Wniosek: Odrzucamy H0.
Zmienne PYT_2 i CZY_KIER są ZALEŻNE.


# zadanie 9

In [6]:
def goodman_kruskal_gamma(x, y):
    """
    Oblicza współczynnik Gamma dla dwóch wektorów zmiennych porządkowych.
    """
    crosstab = pd.crosstab(x, y)

    nc = 0  # Pary zgodne
    nd = 0  # Pary niezgodne

    for i in range(crosstab.shape[0]):
        for j in range(crosstab.shape[1]):
            # Pary zgodne: suma wartości na prawo i w dół
            nc += crosstab.iloc[i, j] * crosstab.iloc[i + 1:, j + 1:].sum().sum()
            # Pary niezgodne: suma wartości na lewo i w dół
            nd += crosstab.iloc[i, j] * crosstab.iloc[i + 1:, :j].sum().sum()

    if (nc + nd) == 0:
        return 0  # zabezpieczenie przed dzieleniem przez zero

    return (nc - nd) / (nc + nd)


df_analiza = df.copy()

df_analiza['CZY_KIER_NUM'] = df_analiza['CZY_KIER'].map({'Nie': 0, 'Tak': 1})
df_analiza = df_analiza.dropna(subset=['PYT_2', 'STAŻ', 'CZY_KIER_NUM'])

pary = [
    ('PYT_2', 'CZY_KIER_NUM', 'Zadowolenie vs Stanowisko'),
    ('PYT_2', 'STAŻ', 'Zadowolenie vs Staż pracy'),
    ('CZY_KIER_NUM', 'STAŻ', 'Stanowisko vs Staż pracy')
]

wyniki = []

for var1, var2, opis in pary:
    x = df_analiza[var1]
    y = df_analiza[var2]

    tau, p_val_tau = scipy.stats.kendalltau(x, y)

    gamma = goodman_kruskal_gamma(x, y)

    wyniki.append({
        'Badana relacja': opis,
        'Zmienne': f"{var1} & {var2}",
        'Kendall Tau-b': round(tau, 4),
        'Gamma': round(gamma, 4),
        'p-value (dla Tau)': round(p_val_tau, 4)
    })

df_wyniki = pd.DataFrame(wyniki)
print("--- Miary współzmienności dla zmiennych porządkowych ---")
print(df_wyniki.to_string(index=False))

--- Miary współzmienności dla zmiennych porządkowych ---
           Badana relacja              Zmienne  Kendall Tau-b   Gamma  p-value (dla Tau)
Zadowolenie vs Stanowisko PYT_2 & CZY_KIER_NUM        -0.0130 -0.0341             0.8486
Zadowolenie vs Staż pracy         PYT_2 & STAŻ         0.0481  0.0908             0.4672
 Stanowisko vs Staż pracy  CZY_KIER_NUM & STAŻ         0.2816  0.7527             0.0000


# zadanie dodatkowe 1

In [7]:
def distance_correlation(X, Y):
    X = np.atleast_1d(X)
    Y = np.atleast_1d(Y)

    n = len(X)

    a = np.abs(X[:, None] - X[None, :])
    b = np.abs(Y[:, None] - Y[None, :])

    A = a - a.mean(axis=0)[None, :] - a.mean(axis=1)[:, None] + a.mean()
    B = b - b.mean(axis=0)[None, :] - b.mean(axis=1)[:, None] + b.mean()

    dcov2_xy = np.sum(A * B) / (n ** 2)
    dcov2_xx = np.sum(A * A) / (n ** 2)
    dcov2_yy = np.sum(B * B) / (n ** 2)

    if dcov2_xx > 0 and dcov2_yy > 0:
        dcor = np.sqrt(np.maximum(dcov2_xy, 0) / np.sqrt(dcov2_xx * dcov2_yy))
    else:
        dcor = 0.0

    return dcor


def dcor_test(X, Y, num_permutations=1000):
    """
    Wylicza p-value dla testu niezależności opartego na korelacji odległości.
    """
    obs_dcor = distance_correlation(X, Y)

    count_greater = 0
    Y_permuted = np.copy(Y)

    for _ in range(num_permutations):
        np.random.shuffle(Y_permuted)
        perm_dcor = distance_correlation(X, Y_permuted)

        if perm_dcor >= obs_dcor:
            count_greater += 1

    p_value = (count_greater + 1) / (num_permutations + 1)

    return obs_dcor, p_value


np.random.seed(42)  # dla powtarzalności wyników
n_samples = 100

print("=== SCENARIUSZ 1: Zmienne CAŁKOWICIE NIEZALEŻNE ===")
X_indep = np.random.normal(0, 1, n_samples)
Y_indep = np.random.normal(0, 1, n_samples)

dcor_val, p_val = dcor_test(X_indep, Y_indep)

print(f"Korelacja odległości: {dcor_val:.4f}")
print(f"p-value: {p_val:.4f}")
if p_val < 0.05:
    print("Wniosek: Odrzucamy H0 (Są zależne!)")
else:
    print("Wniosek: Brak podstaw do odrzucenia H0 (Są niezależne - uff, wszystko działa!)")

print("\n=== SCENARIUSZ 2: Zmienne ZALEŻNE NIELINIOWO (Parabola) ===")
X_dep = np.random.uniform(-3, 3, n_samples)
Y_dep = X_dep ** 2 + np.random.normal(0, 0.5, n_samples)

dcor_val2, p_val2 = dcor_test(X_dep, Y_dep)

print(f"Korelacja odległości: {dcor_val2:.4f}")
print(f"p-value: {p_val2:.4f}")

pearson_corr = np.corrcoef(X_dep, Y_dep)[0, 1]
print(f"Dla porównania, zwykła korelacja Pearsona wynosi: {pearson_corr:.4f} (bliska 0!)")

if p_val2 < 0.05:
    print("Wniosek: Odrzucamy H0 (Są zależne - dCor pięknie wyłapał nieliniowość!)")
else:
    print("Wniosek: Brak podstaw do odrzucenia H0.")

=== SCENARIUSZ 1: Zmienne CAŁKOWICIE NIEZALEŻNE ===
Korelacja odległości: 0.1903
p-value: 0.2507
Wniosek: Brak podstaw do odrzucenia H0 (Są niezależne - uff, wszystko działa!)

=== SCENARIUSZ 2: Zmienne ZALEŻNE NIELINIOWO (Parabola) ===
Korelacja odległości: 0.5061
p-value: 0.0010
Dla porównania, zwykła korelacja Pearsona wynosi: -0.1250 (bliska 0!)
Wniosek: Odrzucamy H0 (Są zależne - dCor pięknie wyłapał nieliniowość!)
